<a href="https://colab.research.google.com/github/ritaireriy-wz/AUS/blob/main/Data_Preprocessing_%26_Custom_Feature_Engineering_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: High-Performance Data Preprocessing & Custom Feature Engineering Pipeline

This system implements a highly optimized suite of foundational vectorized data preprocessing and array manipulation utilities using NumPy. These modules form the backbone of custom production pipelines for feature normalization, binning, and searching.

### Project Objective

The primary objective of this project is to develop a lightweight, production-ready library of vectorized array operations to bypass high-level framework overhead. The toolset targets critical preprocessing tasks, such as dynamic normalization, target indexing, and feature binning.

### Key Constraints & Design Principles:
1. **Zero High-Level Dependencies**: Implement calculations using base NumPy routines, avoiding high-level frameworks like Pandas or Scikit-learn.
2. **Vectorization-First**: Eliminate native Python loops and iterative conditioning to leverage SIMD vector hardware acceleration and optimized memory layouts.
3. **Arbitrary Dimensionality**: Ensure scaling and standardization routines seamlessly support multi-dimensional tensors across configurable axes using NumPy broadcasting.

## Module 1: Preprocessing & Vectorized Operations

In [8]:
import numpy as np

---

### Section 1.1: Feature Min-Max Rescaling

#### Feature Engineering Specifications

Implement and test linear min-max scaling pipelines for robust scaling across arbitrary ranges.

#### Core Component 1: Global Array Rescaling

Implement `minmax_rescale(arr, a, b)` to linearly map any arbitrary n-dimensional array globally into a target interval $[a, b]$ where the global minimum of the dataset maps to $a$, and the global maximum maps to $b$.

In [12]:
import numpy as np

def minmax_rescale(arr, a, b):
    arr_min = np.min(arr)
    arr_max = np.max(arr)
    if arr_max == arr_min:
        return np.full_like(arr, a)
    return a + ((arr - arr_min) * (b - a)) / (arr_max - arr_min)

# Test case
arr = np.array([[0.5, 1.2], [2.2, 1.0], [1.3, 0.9]])
arr1 = minmax_rescale(arr, -2., 3.)
assert np.allclose(arr1, np.array([[-2., 0.0588], [3., -0.529], [0.352, -0.823]]), atol=0.01)
print("Exercise 5.1, Task 1 passed!")

Exercise 5.1, Task 1 passed!


#### Core Component 2: Variable-wise Independent Scaling

Implement `minmax_rescale_per_var(arr, a, b)` to rescale slice elements independently across axis #1. For any output matrix $R$, each independent column slice $R[i, :]$ must strictly contain a local minimum of $a$ and a local maximum of $b$.

In [22]:
import numpy as np

def minmax_rescale_per_var(arr, a, b):
    # Calculate scale parameters along axis=1 to match the test case expectations
    row_min = np.min(arr, axis=1, keepdims=True)
    row_max = np.max(arr, axis=1, keepdims=True)
    diff = row_max - row_min
    diff_replaced = np.where(diff == 0, 1.0, diff)
    rescaled = a + ((arr - row_min) * (b - a)) / diff_replaced
    return np.where(diff == 0, a, rescaled)

# Test case
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
arr1 = minmax_rescale_per_var(arr, -1., 1.)
assert np.allclose(arr1, np.array([[1., 0., -1.], [-1., .6, 1.]]), atol=0.01)
print("Core Component 2: Variable-wise rescaling verified successfully!")

Core Component 2: Variable-wise rescaling verified successfully!


#### Core Component 3: Generalized Arbitrary Axis Rescaling

Implement a highly optimized, single-pass vectorized function `minmax_rescale_along_axis(arr, a, b, axis)` that normalizes data independently across any user-defined axis. Avoid `if-else` branching blocks to maximize computation performance and leverage modern tensor broadcasting rules.

In [15]:
import numpy as np

def minmax_rescale_along_axis(arr, a, b, axis):
    axis_min = np.min(arr, axis=axis, keepdims=True)
    axis_max = np.max(arr, axis=axis, keepdims=True)
    diff = axis_max - axis_min
    diff_replaced = np.where(diff == 0, 1.0, diff)
    rescaled = a + ((arr - axis_min) * (b - a)) / diff_replaced
    return np.where(diff == 0, a, rescaled)

# Test case
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
arr1 = minmax_rescale_along_axis(arr, -1., 1., axis=0)
assert np.allclose(arr1, np.array([[1., 1., -1.], [-1., -1., 1.]]), atol=0.01)
arr1 = minmax_rescale_along_axis(arr, -1., 1., axis=1)
assert np.allclose(arr1, np.array([[1., 0., -1.], [-1., .6, 1.]]), atol=0.01)
print("Core Component 3: Generalized axis rescaling verified successfully!")

Core Component 3: Generalized axis rescaling verified successfully!


---

### Section 1.2: Standard Score Normalization (Z-Score Standardization)

#### Specifications

Implement `standardize_along_axis(arr, axis)` to dynamically standardize 2D datasets along any specified dimension, centering the features to a mean of $0$ and scaling to a standard deviation of $1$ for machine learning model ingestion.

In [16]:
import numpy as np

def standardize_along_axis(arr, axis):
    mean = np.mean(arr, axis=axis, keepdims=True)
    std = np.std(arr, axis=axis, keepdims=True)
    std_replaced = np.where(std == 0, 1.0, std)
    return (arr - mean) / std_replaced

# Test case
arr = np.array([[2., 1., 0.], [-2., 0., 0.5]])
arr1 = standardize_along_axis(arr, axis=0)
assert np.allclose(arr1, np.array([[1., 1., -1.], [-1., -1., 1.]]), atol=0.01)
arr1 = standardize_along_axis(arr, axis=1)
assert np.allclose(arr1, np.array([[1.224, 0., -1.224], [-1.388, 0.462, 0.925]]), atol=0.01)
print("Section 1.2: Standard Score Normalization verified successfully!")

Section 1.2: Standard Score Normalization verified successfully!


---

### Section 1.3: Fast Custom Target-Search Vector Routing

#### Specifications

Implement `index_of_1d(arr, vals_to_find, not_found_value)` to perform rapid, vectorized search routing for queries. If a value has duplicates, route to the first occurrence index. When a search query is missing, return a user-configurable default value.

In [17]:
import numpy as np

def index_of_1d(arr, vals_to_find, not_found_value):
    mask = (arr == vals_to_find[..., np.newaxis])
    has_match = np.any(mask, axis=-1)
    match_idx = np.argmax(mask, axis=-1)
    return np.where(has_match, match_idx, not_found_value)

# Test cases
arr = np.array([1, 5, 10, 5, 3, 6])
vals_to_find = np.array([[10, 2], [5, 6], [6, 15]])
ret = index_of_1d(arr, vals_to_find, not_found_value=arr.shape[0])
assert np.array_equal(ret, np.array([[2, 6], [1, 5], [5, 6]]))
print("Section 1.3: Target-Search Vector Routing verified successfully!")

Section 1.3: Target-Search Vector Routing verified successfully!


---

### Section 1.4: Fully Vectorized Custom Binning & Frequency Histograms

#### Specifications

Develop a custom, scratch-built histogram generator `histogram(arr, n_bins)` without relying on existing higher-level library utilities. The function must compute equally spaced bin limits and efficiently count multi-dimensional element distributions across the boundaries.

In [18]:
import numpy as np

def histogram(arr, n_bins):
    flat = arr.ravel()
    amin, amax = np.min(flat), np.max(flat)
    if amin == amax:
        bin_edges = np.linspace(amin - 0.5, amin + 0.5, n_bins + 1)
    else:
        bin_edges = np.linspace(amin, amax, n_bins + 1)
    elem_cols = flat[:, np.newaxis]
    greater_equal = (elem_cols >= bin_edges[:-1])
    less_than = (elem_cols < bin_edges[1:])
    is_equal_to_max = (elem_cols == bin_edges[-1])
    in_bin = greater_equal & less_than
    in_bin[:, -1] = in_bin[:, -1] | is_equal_to_max[:, -1]
    hist = np.sum(in_bin, axis=0)
    return hist, bin_edges

# Test cases
arr1 = np.array([0., 3., 3.5, 1.5, 2., 4.])
hist1, bin_edges1 = histogram(arr1, n_bins=2)
assert np.all(hist1 == [2, 4])
assert np.allclose(bin_edges1, [0., 2., 4.])

arr2 = np.array([[10., 11.], [5., 15.], [6., 14.]])
hist2, bin_edges2 = histogram(arr2, n_bins=4)
assert np.all(hist2 == [2, 0, 2, 2])
assert np.allclose(bin_edges2, [5., 7.5, 10., 12.5, 15.])

arr3 = np.array([[[-10.], [-8.]]])
hist3, bin_edges3 = histogram(arr3, n_bins=1)
assert np.all(hist3 == [2])
assert np.allclose(bin_edges3, [-10., -8.])
print("Section 1.4: Primary Histogram implementation verified successfully!")

Section 1.4: Primary Histogram implementation verified successfully!


Try to solve it with another approach. See hints below!

#### Hints for additional approaches

- _(Less efficient, than the second one, but includes broadcasting where everything is compared with everything.)_ The bin edges can be created by, for example, `np.linspace()`. Compare all elements with (almost) all bin edges. You have to count the True values for each element to get bin indices for each element. Then, count the frequency of bin indices (e.g., with `np.bincount()`). You may need to flatten (reshape to 1D) the array at one point.
- As the bin widths are identical (except for the last one), with some prior preparations, a floor division will provide the bin indices.
- A variant of Approach \#1, but not using `np.bincount()`: If we compare all elements with (almost) all bin edges, counting the True values in this mask for each bin (instead of for each element) will give you a cumulative version of the histogram. You can take the differences between the adjacent cumulative bin sizes to get the histogram.

#### Additional approach

In [19]:
import numpy as np

def histogram_alt(arr, n_bins):
    flat = arr.ravel()
    amin, amax = np.min(flat), np.max(flat)
    if amin == amax:
        bin_edges = np.linspace(amin - 0.5, amin + 0.5, n_bins + 1)
    else:
        bin_edges = np.linspace(amin, amax, n_bins + 1)
    if amin == amax:
        idx = np.zeros_like(flat, dtype=int)
    else:
        width = (amax - amin) / n_bins
        idx = ((flat - amin) / width).astype(int)
        idx = np.clip(idx, 0, n_bins - 1)
    hist = np.bincount(idx, minlength=n_bins)
    return hist, bin_edges

# Test cases with alternative
arr1 = np.array([0., 3., 3.5, 1.5, 2., 4.])
hist1, bin_edges1 = histogram_alt(arr1, n_bins=2)
assert np.all(hist1 == [2, 4])
assert np.allclose(bin_edges1, [0., 2., 4.])

arr2 = np.array([[10., 11.], [5., 15.], [6., 14.]])
hist2, bin_edges2 = histogram_alt(arr2, n_bins=4)
assert np.all(hist2 == [2, 0, 2, 2])
assert np.allclose(bin_edges2, [5., 7.5, 10., 12.5, 15.])

arr3 = np.array([[[-10.], [-8.]]])
hist3, bin_edges3 = histogram_alt(arr3, n_bins=1)
assert np.all(hist3 == [2])
assert np.allclose(bin_edges3, [-10., -8.])
print("Section 1.4: Alternative Histogram implementation verified successfully!")

Section 1.4: Alternative Histogram implementation verified successfully!
